In [6]:
import pandas as pd
import numpy as np
import glob

In [7]:
df = pd.read_csv('Boulder_20231107_Mayor.csv', dtype=str)
df.head()

,BallotType,BatchId,CvrNumber,ImprintedId,RecordId,TabulatorNum,rank1,rank2,rank3,rank4
0,DS-01,1.0,1,108-1-104,104.0,108.0,Brockett,Speer,Yates,Tweedlie
1,DS-01,1.0,2,108-1-103,103.0,108.0,Yates,Tweedlie,Brockett,Speer
2,DS-01,1.0,3,108-1-66,66.0,108.0,Yates,Tweedlie,skipped,skipped
3,DS-01,1.0,4,108-1-65,65.0,108.0,Yates,skipped,skipped,skipped
4,DS-01,1.0,5,108-1-42,42.0,108.0,Yates,skipped,skipped,skipped


In [6]:
import os
os.getcwd()

'C:\\Users\\rchol'

In [8]:
rank_cols = ['rank1', 'rank2', 'rank3', 'rank4']
df_clean = df[df[rank_cols].notna().all(axis = 1)]

def clean_ballot(row):
    ranked = []
    for c in row:
        if c in ["undervote", "skipped", "overvote", "writein"]:
            break
        ranked.append(c)
    return ranked

ballots = [(list(row[6:10]), 1) for row in df_clean.values.tolist()]

candidates = set()
for ballot, weight in ballots:
    for candidate in ballot:
        candidates.add(candidate)
candidates = list(candidates)

print(f"Number of Ballots: {len(ballots)}")
print(f"Candidates: {candidates}")


Number of Ballots: 118787
Candidates: ['Brockett', 'Speer', 'Yates', 'Tweedlie', 'skipped']


In [9]:
df_clean.to_csv('cleaned_Boulder_20231107_Mayor.csv', index=False)

In [10]:
df_clean.head()

,BallotType,BatchId,CvrNumber,ImprintedId,RecordId,TabulatorNum,rank1,rank2,rank3,rank4
0,DS-01,1.0,1,108-1-104,104.0,108.0,Brockett,Speer,Yates,Tweedlie
1,DS-01,1.0,2,108-1-103,103.0,108.0,Yates,Tweedlie,Brockett,Speer
2,DS-01,1.0,3,108-1-66,66.0,108.0,Yates,Tweedlie,skipped,skipped
3,DS-01,1.0,4,108-1-65,65.0,108.0,Yates,skipped,skipped,skipped
4,DS-01,1.0,5,108-1-42,42.0,108.0,Yates,skipped,skipped,skipped


In [11]:
df = pd.read_csv('Boulder_20231107_Mayor.csv')

C:\Users\rchol\AppData\Local\Temp\ipykernel_38792\2658530270.py:1: DtypeWarning: Columns (1,2,4,5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('Boulder_20231107_Mayor.csv')


In [12]:
first_column = df_clean.columns[9]

number_counts = df_clean[first_column].value_counts()

print(f"Number of unique values in the first column: {len(number_counts)}")
print(number_counts)

Number of unique values in the first column: 5
rank4
skipped     105200
Tweedlie      8965
Yates         2018
Speer         1498
Brockett      1106
Name: count, dtype: int64


In [13]:
ballots_df = pd.read_csv('cleaned_Boulder_20231107_Mayor.csv')

C:\Users\rchol\AppData\Local\Temp\ipykernel_38792\2793734122.py:1: DtypeWarning: Columns (1,2,4,5) have mixed types. Specify dtype option on import or set low_memory=False.
  ballots_df = pd.read_csv('cleaned_Boulder_20231107_Mayor.csv')


In [14]:
# Convert each row into a tuple: (preferences list, weight)
ballots = [(clean_ballot(row[6:10]), 1) for row in df_clean.values.tolist()]

# Identify all unique candidates (from the ranked preferences)
candidates = set()
for ballot, weight in ballots:
    for candidate in ballot:
        if candidate not in {"skipped", "overvote", "undervote", "writein"}:
            candidates.add(candidate)
candidates = list(candidates)

def count_votes(ballots, candidates):
    """
    Count the votes for the current round using weighted ballots.
    Each ballot contributes its weight to the vote of the highest-ranked candidate
    that hasn't been eliminated.
    """
    vote_counts = {candidate: 0 for candidate in candidates}
    for ballot, weight in ballots:
        for candidate in ballot:
            if candidate in candidates:
                vote_counts[candidate] += weight
                break  # Only count the top remaining candidate.
    return vote_counts

round_num = 1
while True:
    # Tally the weighted votes for this round.
    vote_counts = count_votes(ballots, candidates)
    total_votes = sum(vote_counts.values())
    print(f"Round {round_num} vote counts: {vote_counts} (Total active votes: {total_votes})")

    # Check if any candidate has a majority (>50% of active votes)
    for candidate, count in vote_counts.items():
        if count > total_votes / 2:
            print(f"\nWinner: {candidate} with {count} votes (majority of {total_votes} active votes)!")
            winner = candidate
            break
    else:
        # No candidate has a majority; eliminate candidate(s) with the fewest votes.
        min_votes = min(vote_counts.values())
        eliminated = [candidate for candidate, count in vote_counts.items() if count == min_votes]
        print(f"Eliminated in round {round_num}: {eliminated}\n")
         # Remove the eliminated candidate(s) from further consideration.
        for candidate in eliminated:
            candidates.remove(candidate)
        
        # If no candidates remain or all remaining are tied, declare a tie.
        if not candidates or len(vote_counts) == len(eliminated):
            print("Election resulted in a tie among the remaining candidates:")
            print(vote_counts)
            break

        round_num += 1
        continue
    break

Round 1 vote counts: {'Yates': 14274, 'Brockett': 11540, 'Tweedlie': 749, 'Speer': 6371} (Total active votes: 32934)
Eliminated in round 1: ['Tweedlie']

Round 2 vote counts: {'Yates': 14660, 'Brockett': 11656, 'Speer': 6489} (Total active votes: 32805)
Eliminated in round 2: ['Speer']

Round 3 vote counts: {'Yates': 15596, 'Brockett': 16865} (Total active votes: 32461)

Winner: Brockett with 16865 votes (majority of 32461 active votes)!


In [15]:
from itertools import combinations
candidates = set()
for ballot, weight in ballots:
    for candidate in ballot:
        if candidate not in {"skipped", "overvote", "undervote", "writein"}:
            candidates.add(candidate)
candidates = list(candidates)
print(len(candidates), candidates)
def simulate_copeland_fast(ballots, candidates):
    cand_index = {c: i for i, c in enumerate(candidates)}
    n = len(candidates)
    matrix = np.zeros((n, n))
    
    for ballot, weight in ballots:
        ranked = [c for c in ballot if c in cand_index]
        unranked = [c for c in candidates if c not in ranked]
        
        # Ranked candidates vs each other: order on the ballot determines the win
        for c1, c2 in combinations(ranked, 2):
            matrix[cand_index[c1]][cand_index[c2]] += weight
        
        # Every ranked candidate beats every unranked candidate
        for c1 in ranked:
            for c2 in unranked:
                matrix[cand_index[c1]][cand_index[c2]] += weight
        
        # Unranked vs unranked: no preference was expressed, so skip (tie)
    
    scores = {c: 0 for c in candidates}
    for c1, c2 in combinations(candidates, 2):
        i, j = cand_index[c1], cand_index[c2]
        if matrix[i][j] > matrix[j][i]:
            scores[c1] += 1
            scores[c2] -= 1
        elif matrix[j][i] > matrix[i][j]:
            scores[c2] += 1
            scores[c1] -= 1
    sorted_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    print("\nCopeland Scores:")
    for candidate, score in sorted_scores:
        print(f"  {candidate}: {score}")
    winner = sorted_scores[0][0]
    print(f"\nCopeland Winner: {winner}")
    return winner

4 ['Yates', 'Brockett', 'Tweedlie', 'Speer']


In [16]:
copeland_winner = simulate_copeland_fast(ballots, candidates)


Copeland Scores:
  Brockett: 3
  Yates: 1
  Speer: -1
  Tweedlie: -3

Copeland Winner: Brockett


In [24]:
def simulate_actual_plurality_veto(ballots, candidates):
    cand_set = set(candidates)

    scores = {c: 0 for c in candidates}
    expanded_voters = []

    for ballot, weight in ballots:
        ranked = [c for c in ballot if c in cand_set]
        if ranked:
            scores[ranked[0]] += weight
            for _ in range(weight):
                expanded_voters.append(ranked)

    standing = set(candidates)
    elimination_order = []

    for ranked_ballot in expanded_voters:
        if len(standing) == 1:
            break

        ranked_set = set(ranked_ballot)
        # Every standing candidate this voter left off their ballot is treated as equally "last" and gets vetoed simultaneously.
        unranked_standing = standing - ranked_set

        vetoed_this_pass = set(unranked_standing)

        if not unranked_standing:
            for c in reversed(ranked_ballot):
                if c in standing:
                    vetoed_this_pass = {c}
                    break

        for vetoed in vetoed_this_pass:
            scores[vetoed] -= 1
            if scores[vetoed] <= 0 and vetoed in standing:
                standing.remove(vetoed)
                elimination_order.append(vetoed)

    if standing:
        winner = list(standing)[0]
    else:
        winner = elimination_order[-1]
    print(f"Elimination order: {elimination_order}")
    print(f"Plurality Veto Winner: {winner}")
    return winner, elimination_order

In [25]:
plurality_veto_winner, plurality_veto_elimination_order = simulate_actual_plurality_veto(ballots, candidates)

Elimination order: ['Tweedlie', 'Speer', 'Brockett']
Plurality Veto Winner: Yates


In [19]:
last_place_counts = {
    'Tweedlie' : 8965,
    'Yates' : 2018,
    'Speer': 1496,
    'Brockett' : 1106
}
def simulate_actual_plurality_veto(ballots, candidates, last_place_counts):
    cand_set = set(candidates)

    scores = {c: 0 for c in candidates}
    expanded_voters = []

    for ballot, weight in ballots:
        ranked = [c for c in ballot if c in cand_set]
        if ranked:
            scores[ranked[0]] += weight
            for _ in range(weight):
                expanded_voters.append(ranked)

    # Derive the denominator from the counts themselves 
    # total last-place appearances across all candidates = total complete-ballot weight
    total_last_place = sum(last_place_counts.get(c, 0) for c in candidates)

    if total_last_place > 0:
        last_place_pct = {
            c: last_place_counts.get(c, 0) / total_last_place
            for c in candidates
        }
    else:
        last_place_pct = {c: 0.0 for c in candidates}

    standing = set(candidates)
    elimination_order = []

    for ranked_ballot in expanded_voters:
        if len(standing) == 1:
            break

        ranked_set = set(ranked_ballot)
        unranked_standing = standing - ranked_set

        vetoed_this_pass = set(unranked_standing)

        if not unranked_standing:
            for c in reversed(ranked_ballot):
                if c in standing:
                    vetoed_this_pass = {c}
                    break

        newly_eliminated = []
        for vetoed in vetoed_this_pass:
            scores[vetoed] -= 1
            if scores[vetoed] <= 0 and vetoed in standing:
                newly_eliminated.append(vetoed)

        newly_eliminated.sort(key=lambda c: (-last_place_pct[c], c))

        for c in newly_eliminated:
            if c in standing:
                standing.remove(c)
                elimination_order.append(c)

    if standing:
        winner = list(standing)[0]
    else:
        winner = elimination_order[-1]
    print(f"Elimination order: {elimination_order}")
    print(f"Plurality Veto Winner: {winner}")
    return winner, elimination_order

In [20]:
winner, elimination_order = simulate_actual_plurality_veto(ballots, candidates, last_place_counts)

Elimination order: ['Tweedlie', 'Speer', 'Brockett']
Plurality Veto Winner: Yates
